# 01 — Web Scraping: Papal Encyclicals Corpus

**DS 5001 — Exploratory Text Analytics Final Project**  
**Source:** https://www.papalencyclicals.net/document-directory

This notebook walks through scraping the papal encyclicals corpus from
papalencyclicals.net. The site organizes documents by pope, with each
encyclical linked to a page containing the full text (usually HTML, sometimes epub).

## Steps
1. Scrape the document directory to build an index of all encyclicals
2. Download the full text of each encyclical
3. Save raw text files and metadata

In [1]:
import sys
sys.path.insert(0, '..')

import os
os.environ["SCRAPER_ALLOW_INSECURE_SSL"] = "true"

import importlib
import src.scraper as scraper
scraper = importlib.reload(scraper)  # always pick up latest scraper.py edits

# Rebind commonly used symbols from the reloaded module
scrape_index = scraper.scrape_index
scrape_documents = scraper.scrape_documents
save_index = scraper.save_index
save_library_csv = scraper.save_library_csv
fetch_page = scraper.fetch_page
parse_directory = scraper.parse_directory
INDEX_FILE = scraper.INDEX_FILE
RAW_DIR = scraper.RAW_DIR
DATA_DIR = scraper.DATA_DIR

import json
import pandas as pd
from pathlib import Path

2026-03-27 11:21:02,067 [INFO] NumExpr defaulting to 8 threads.


In [2]:
# Check optional dependencies
import importlib.util

def _check_dep(package, install_cmd):
    if importlib.util.find_spec(package) is None:
        print(f"'{package}' is NOT installed — some features will be disabled.")
        print(f"   To enable: {install_cmd}")
    else:
        print(f"'{package}' is installed.")

_check_dep("lxml",     "pip install lxml")
_check_dep("ebooklib", "pip install ebooklib")


'lxml' is installed.
'ebooklib' is installed.


## Step 1: Scrape the Document Directory

The directory page lists encyclicals organized by pope. We parse it to
extract document titles, URLs, and associated pope names.

In [3]:
# Scrape the directory index (or load if already scraped)
# Set to True after scraper logic changes to rebuild clean metadata
FORCE_REBUILD_INDEX = True

if INDEX_FILE.exists() and not FORCE_REBUILD_INDEX:
    with open(INDEX_FILE) as f:
        documents = json.load(f)
    print(f"Loaded existing index: {len(documents)} documents")
else:
    documents = scrape_index()
    save_index(documents)
    print(f"Scraped index: {len(documents)} documents")

2026-03-27 11:21:02,349 [INFO] Fetching document directory from https://www.papalencyclicals.net/document-directory
2026-03-27 11:21:02,406 [INFO] SSL verification failed in this environment; using insecure fallback (verify=False).
2026-03-27 11:21:04,248 [INFO] Parsed 570 unique documents from 46 sections
2026-03-27 11:21:04,265 [INFO] Saved index with 570 documents to C:\Users\harrisrc\OneDrive - Chesterfield County VA\Documents\MSDS\encyclicals\data\encyclicals_index.json


Scraped index: 570 documents


In [4]:
# Preview the index
import numpy as np

df_index = pd.DataFrame(documents)

# Backfill normalized fields for legacy index rows loaded from disk.
if 'category' not in df_index.columns:
    df_index['category'] = np.where(
        df_index.get('pope', '').fillna('').str.lower().eq('church councils'),
        'council',
        'pope'
    )
if 'author' not in df_index.columns:
    df_index['author'] = np.where(
        df_index['category'].eq('council'),
        df_index.get('title', ''),
        df_index.get('pope', '')
    )
if 'author_dates' not in df_index.columns:
    df_index['author_dates'] = np.where(
        df_index['category'].eq('council'),
        '',
        df_index.get('pope_dates', '')
    )

print('Documents by category:')
print(df_index['category'].value_counts())

print('\nTop authors:')
print(df_index['author'].value_counts().head(20))

# Years can be empty strings or malformed; coerce to numeric before range stats.
years = pd.to_numeric(df_index['year'], errors='coerce')
if years.notna().any():
    print(f"\nYears covered (valid only): {int(years.min())} - {int(years.max())}")
else:
    print('\nYears covered (valid only): n/a')

print(f"Missing/invalid year values: {years.isna().sum()} of {len(df_index)}")

Documents by category:
category
pope       549
council     21
Name: count, dtype: int64

Top authors:
author
Pope Leo XIII            88
Pope Pius XII            60
Pope St. John Paul II    60
Pope Benedict XIV        44
Pope Bl. Pius IX         42
Pope Paul VI             36
Pope Pius XI             32
Pope St. Pius X          26
Pope Pius VI             26
Pope Benedict XVI        21
Pope St. John XXIII      18
Pope Clement XIII        13
Pope Benedict XV         12
Pope Gregory XVI         10
Pope Francis             10
Pope Leo XII              5
Pope Alexander IV         4
Pope Clement XIV          4
Pope John XXII            4
Pope St. Pius V           4
Name: count, dtype: int64

Years covered (valid only): 1215 - 2016
Missing/invalid year values: 557 of 570


In [5]:
# Audit source domains to troubleshoot external links (e.g., digilander)
from urllib.parse import urlparse

df_domains = df_index.copy()
df_domains['domain'] = df_domains['url'].fillna('').apply(lambda u: urlparse(u).netloc.lower())
print('Top source domains:')
print(df_domains['domain'].value_counts().head(20))

external = df_domains[~df_domains['domain'].str.contains('papalencyclicals.net|vatican.va', regex=True, na=False)]
print(f"\nExternal-domain records: {len(external)}")
if len(external):
    print(external[['pope', 'title', 'url']].head(25).to_string(index=False))

Top source domains:
domain
www.papalencyclicals.net      401
www.vatican.va                 92
digilander.iol.it              31
w2.vatican.va                  27
web.archive.org                 7
www.franciscan-archive.org      5
www.bluewaterarts.com           1
en.wikisource.org               1
www.ewtn.com                    1
www.nativeweb.org               1
www.cin.org                     1
digilander.libero.it            1
www.adoremus.org                1
Name: count, dtype: int64

External-domain records: 50
                 pope                                                          title                                                                                                                                                                             url
    Pope Alexander IV                                         Clara claris praeclara                                                                                                                          http://ww

In [ ]:
# Build a dead-link replacement worklist from known dead domains
# (kept in sync with scraper.py)
dead_domains = set(scraper.KNOWN_DEAD_DOMAINS)

dead_links = df_domains[df_domains['domain'].isin(dead_domains)].copy()
print(f"Dead-link records: {len(dead_links)}")
if len(dead_links):
    cols = ['doc_id', 'category', 'author', 'author_dates', 'pope', 'pope_dates', 'title', 'url', 'year', 'document_type', 'language', 'format', 'domain']
    print(dead_links[cols].head(25).to_string(index=False))

    worklist = DATA_DIR / 'processed' / 'dead_link_replacements.csv'
    dead_links.assign(replacement_url='').to_csv(worklist, index=False)
    print(f"\nSaved replacement worklist: {worklist}")

Dead-link records: 32
                                   doc_id              pope                    title                                                url year               domain
 pope_clement_xiii__accedamus_cum_fiducia Pope Clement XIII    Accedamus cum fiducia    http://digilander.iol.it/magistero/c13acced.htm         digilander.iol.it
    pope_clement_xiii__pastoralis_officii Pope Clement XIII       Pastoralis officii    http://digilander.iol.it/magistero/c13pasto.htm         digilander.iol.it
         pope_clement_xiii__quam_graviter Pope Clement XIII            Quam graviter    http://digilander.iol.it/magistero/c13quamg.htm         digilander.iol.it
        pope_clement_xiii__quanta_auxilii Pope Clement XIII           Quanta auxilii    http://digilander.iol.it/magistero/c13quant.htm         digilander.iol.it
      pope_clement_xiii__quanto_in_dolore Pope Clement XIII         Quanto in dolore    http://digilander.iol.it/magistero/c13quaid.htm         digilander.iol.it
  pope

## Step 2: Download Document Texts

For each document in the index, we fetch the linked page and extract
the encyclical text. The scraper handles:
- HTML pages with text in the page body
- Pages that link to epub files
- Language detection (English, Latin, Italian, French)

In [7]:
# Scrape all document texts (skips already-downloaded files)
# Use max_docs to limit for testing:
# documents = scrape_documents(documents, max_docs=100, show_progress=True, progress_label="Downloading")

documents = scrape_documents(documents, show_progress=True, progress_label="Downloading")
save_index(documents)
save_library_csv(documents)

Downloading: 341/570 | saved=9 skipped=325 dead=7 errors=0

2026-03-27 11:21:44,296 [INFO] Failed to fetch http://www.cin.org/v2east.html after 3 attempts


Downloading: 570/570 | saved=9 skipped=528 dead=32 errors=1

2026-03-27 11:21:47,852 [INFO] Saved index with 570 documents to C:\Users\harrisrc\OneDrive - Chesterfield County VA\Documents\MSDS\encyclicals\data\encyclicals_index.json
2026-03-27 11:21:47,858 [INFO] Saved LIBRARY.csv with 570 rows


In [8]:
# Summary
df = pd.DataFrame(documents)
print(f"Total documents: {len(df)}")
print(f"\nBy language:")
print(df['language'].value_counts())
print(f"\nBy format:")
print(df['format'].value_counts())
print(f"\nDocuments with text: {(df['text_length'] > 100).sum()}")

Total documents: 570

By language:
language
en         560
unknown      9
             1
Name: count, dtype: int64

By format:
format
html     569
error      1
Name: count, dtype: int64

Documents with text: 528


In [9]:
# List raw text files
raw_files = list(RAW_DIR.glob('*.txt'))
print(f"Raw text files on disk: {len(raw_files)}")
sizes = [f.stat().st_size for f in raw_files]
print(f"Total size: {sum(sizes)/1024/1024:.1f} MB")
print(f"Average size: {sum(sizes)/len(sizes)/1024:.1f} KB")

Raw text files on disk: 567
Total size: 20.6 MB
Average size: 37.1 KB
